# Exploring DOI discrepnacies
Our goal was to better understand when DOIs don't exactly match but records are still duplicates. We investigated different cases of this to further our thinking on implementing additional rules to improve specificity while capturing as many dupes as possible

In [3]:
import sys
from pathlib import Path

import pandas as pd

# below are the different classifiers we will try out

sys.path.insert(0, "..")

In [ ]:
REPO_DIR = Path.cwd().parent
DATA_PATH = REPO_DIR / "notebooks/results/scored_results_algorithm_builder.csv"
RAW_DATA = REPO_DIR / "notebooks/data/srsr_data.csv"

test_train_compared_df = pd.read_csv(DATA_PATH)
raw_df = pd.read_csv(RAW_DATA, encoding="latin-1")

In [ ]:
# Inspect true duplicates with mismatched DOIs
# Only where both records have a DOI (i.e. exclude NaN/missing DOIs)


def both_have_doi(row):
    doi_a = raw_df[raw_df["RecordID"] == row["id_a"]]["DOI"].values
    doi_b = raw_df[raw_df["RecordID"] == row["id_b"]]["DOI"].values
    a_present = len(doi_a) > 0 and pd.notna(doi_a[0]) and str(doi_a[0]).strip() != ""
    b_present = len(doi_b) > 0 and pd.notna(doi_b[0]) and str(doi_b[0]).strip() != ""
    return a_present and b_present


# True duplicates where DOI match score < 1 (i.e. DOIs genuinely differ)
true_dupes_mismatched_doi = test_train_compared_df[
    (test_train_compared_df["is_dupe"] == 1) & (test_train_compared_df["doi"] < 1.0)
].copy()

print(
    f"True duplicates with mismatched DOIs (inc. missing): {len(true_dupes_mismatched_doi)}"
)

# Filter to only pairs where both records have a DOI
both_doi_mask = true_dupes_mismatched_doi.apply(both_have_doi, axis=1)
true_dupes_mismatched_doi = true_dupes_mismatched_doi[both_doi_mask]

raw_cols = [
    "RecordID",
    "Title",
    "Author",
    "Year",
    "DOI",
    "Journal",
    "Abstract",
    "Pages",
    "Volume",
    "Number",
    "ISBN",
    "ISSN",
    "Publisher",
    "Address",
    "Edition",
    "BookTitle",
    "Series",
]
raw_cols_present = [c for c in raw_cols if c in raw_df.columns]

print(
    f"True duplicates with genuinely different DOIs: {len(true_dupes_mismatched_doi)}"
)
print(
    f"DOI score distribution:\n{true_dupes_mismatched_doi['doi'].value_counts().sort_index()}\n"
)

raw_rows = []
for _, row in true_dupes_mismatched_doi.iterrows():
    for side, record_id in [("A", row["id_a"]), ("B", row["id_b"])]:
        raw_record = raw_df[raw_df["RecordID"] == record_id][raw_cols_present]
        if not raw_record.empty:
            row_dict = raw_record.iloc[0].to_dict()
            row_dict["side"] = side
            row_dict["pair_index"] = row.name
            row_dict["doi_score"] = row["doi"]
            raw_rows.append(row_dict)

# Use a different name to avoid overwriting raw_df
results_df = pd.DataFrame(raw_rows)
display_cols = ["pair_index", "side", "doi_score"] + raw_cols_present
display_cols = [c for c in display_cols if c in results_df.columns]
results_df = results_df[display_cols].sort_values(["pair_index", "side"])

Path("notebooks/results").mkdir(exist_ok=True)
results_df.to_csv("notebooks/results/true_dupes_mismatched_doi.csv", index=False)

results_df

True duplicates with mismatched DOIs (inc. missing): 1760
True duplicates with genuinely different DOIs: 92
DOI score distribution:
doi
0.0    92
Name: count, dtype: int64



,pair_index,side,doi_score,RecordID,Title,Author,Year,DOI,Journal,Abstract,Pages,Volume,Number,ISBN
0,309,A,0.0,396,PHOTOBIOMODULATION IN PERIODONTOLOGY AND IMPLA...,Gholami L.Asefi S.Hooshyarfard A.Sculean A.Rom...,2019.0,10.1089/photob.2019.4710,Photobiomodul Photomed Laser Surg,(Part 2 of this article can be located at www....,739-765,37,12,2578-5478
1,309,B,0.0,25435,PHOTOBIOMODULATION IN PERIODONTOLOGY AND IMPLA...,Gholami L.Asefi S.Hooshyarfard A.Sculean A.Rom...,2019.0,10.1089/photob.2019.4731,Photobiomodul Photomed Laser Surg,(Part 1 of this article can be located at www....,766-783,37,12,2578-5478 (electronic)\r\n2578-5478
2,311,A,0.0,25436,PHOTOBIOMODULATION IN PERIODONTOLOGY AND IMPLA...,Gholami L.Asefi S.Hooshyarfard A.Sculean A.Rom...,2019.0,10.1089/photob.2019.4710,Photobiomodul Photomed Laser Surg,(Part 2 of this article can be located at www....,739-765,37,12,2578-5478 (electronic)\r\n2578-5478
3,311,B,0.0,25435,PHOTOBIOMODULATION IN PERIODONTOLOGY AND IMPLA...,Gholami L.Asefi S.Hooshyarfard A.Sculean A.Rom...,2019.0,10.1089/photob.2019.4731,Photobiomodul Photomed Laser Surg,(Part 1 of this article can be located at www....,766-783,37,12,2578-5478 (electronic)\r\n2578-5478
4,313,A,0.0,25435,PHOTOBIOMODULATION IN PERIODONTOLOGY AND IMPLA...,Gholami L.Asefi S.Hooshyarfard A.Sculean A.Rom...,2019.0,10.1089/photob.2019.4731,Photobiomodul Photomed Laser Surg,(Part 1 of this article can be located at www....,766-783,37,12,2578-5478 (electronic)\r\n2578-5478
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,20778,B,0.0,23816,META-ANALYSIS OF RAT LUNG TUMORS FROM LIFETIME...,Valberg P. A.Crouch E. A.,1999.0,10.1289/ehp.99107693,Environ Health Perspect,Estimating the carcinogenic potential of expos...,693-9,107,9,0091-6765 (Print)\r\n0091-6765
180,20790,A,0.0,38213,ASSIMILATION EFFICIENCIES OF CHEMICAL CONTAMIN...,Wang W. X.Fisher N. S.,1999.0,10.1897/1551-5028%281999%29018%3C2034:AEOCCI%3...,Environ Toxicol Chem,Assimilation efficiencies of contaminants from...,2034-2045,18,9,0730-7268
181,20790,B,0.0,57009,ASSIMILATION EFFICIENCIES OF CHEMICAL CONTAMIN...,Wang WXFisher NS,1999.0,10.1002/etc.5620180923,Environ Toxicol Chem,NaN,2034-2045,18,9,NaN
182,21006,A,0.0,38598,NONNEOPLASTIC NASAL LESIONS IN RATS AND MICE,Monticello T. M.Morgan K. T.Uraih L.,1990.0,10.2307/3430688,Environ Health Perspect,Rodents are commonly used for inhalation toxic...,249-274,85,NaN,0091-6765
